# FYOE — train v5 model on Colab T4

**Before you start:** click `Runtime` → `Change runtime type` → set hardware accelerator to **T4 GPU**.

Then `Runtime` → `Run all`. ~25-30 min total. At the end you get `saved_model_v5.zip` to download.

**What changed (v4.3 → v5):**

v4.3 hit 98.9% IID and ~95%+ seed suite on single messages. **v5** adds:

- **Multi-turn conversation context** — the model now sees the last 3-5 messages as context via tokenizer text-pair encoding (`<s> context </s></s> current </s>`)
- **241 multi-turn templates** across 10 banks: context buildup, confirmation, correction, addition, disambiguation, and critical negatives
- **Context buildup** — "bro I'm hungry" → "same" → "dominos?" fires food_order (bare "dominos?" alone = nothing)
- **Confirmation carry-forward** — "should we order pizza?" → "yeah do it" fires food_order
- **Contrastive disambiguation** — "sure" after "order pizza?" = food_order, but "sure" after "that movie was good" = nothing
- **No-carryforward negatives** — casual chat after actionable must NOT inherit the prior intent
- All v4.3 improvements preserved (focal loss, idiomatic negatives, ultra-short boundary, etc.)

Total dataset: ~30K examples (2,486 multi-turn). Seed suite: 126 cases (was 110).

**Goal:** seed-suite ≥95% including multi-turn context tests, IID ≥98%.

## 1. Setup — fresh VM, clone repo, install deps

**Before running:** push your v4 changes (`generate_data.py` import + integration block, `v4_failure_modes.py`) to a branch named `v4` on GitHub. The clone below pulls that branch.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf paychat-model
# Clone v5 branch — fall back to v4 if v5 doesn't exist yet so the notebook still runs.
!git clone -b v5 https://github.com/Akash-Cheerla/paychat-model.git || git clone -b v4 https://github.com/Akash-Cheerla/paychat-model.git
%cd paychat-model
!git log -1 --oneline
!ls training/v5_multiturn.py && echo 'v5 banks present' || echo 'WARNING: v5_multiturn.py missing — push v5 branch first'

In [ ]:
# Pin transformers to last stable 4.x. transformers 5.0 has regressions in checkpoint loading.
!pip install -q 'transformers==4.46.3' 'tokenizers>=0.20,<0.21' sentencepiece scikit-learn
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 2. Generate training data (v5)

~30K examples: v3 positives + v4 failure-mode banks + v5 multi-turn context banks + per-intent hard negatives. The v5 banks teach the model to use conversation context — 241 templates across 10 categories (context buildup, confirmation, correction, disambiguation, etc.).

In [ ]:
%cd /content/paychat-model/training
!python generate_data.py

In [ ]:
# Sanity check: confirm v4 + v5 + v5.1 categories actually landed in the dataset.
import json
from collections import Counter
ds = json.load(open('full_dataset.json'))
v4_cats = Counter(d['category'] for d in ds if d['category'].startswith('v4_'))
v5_cats = Counter(d['category'] for d in ds if d['category'].startswith('v5_'))
v51_cats = Counter(d['category'] for d in ds if d['category'].startswith('v51_'))
n_context = sum(1 for d in ds if d.get('context'))
print(f'Total examples: {len(ds)}')
print(f'Examples with context field: {n_context}')
print(f'\nv4 categories present:')
for c, n in v4_cats.most_common():
    print(f'  {c:<22} {n}')
assert v4_cats, 'v4 categories missing — generate_data.py did not import V4_* banks'
print(f'\nv5 multi-turn categories:')
for c, n in v5_cats.most_common():
    print(f'  {c:<30} {n}')
assert v5_cats, 'v5 categories missing — generate_data.py did not import v5_multiturn banks'
assert n_context > 0, 'No examples have context — multi-turn data generation failed'
print(f'\nv5.1 regression fix categories:')
for c, n in v51_cats.most_common():
    print(f'  {c:<30} {n}')
assert v51_cats, 'v5.1 categories missing — generate_data.py did not import v5_regression_fixes banks'
n_contact = sum(1 for d in ds if d['labels']['contact'] == 1)
print(f'\ncontact positives: {n_contact}  (v3 had ~600; target >=1000 to fix 0% recall)')
print(f'Total: {len(ds)} examples (target: ~42k)')

## 3. Fine-tune RoBERTa-base (v5)

Training improvements over v4.3:
- **Multi-turn context support** — `ChatDataset` detects examples with a `context` field and uses tokenizer text-pair encoding: `tokenizer(context, text)` → `<s> context </s></s> current </s>`. Single-turn examples still work as before.
- **Focal loss** (gamma=2.0) — downweights easy examples
- **Label smoothing** (0.03) — prevents overconfidence
- **Mixed precision (fp16)** — 2x faster on T4
- **Finer threshold grid** — 0.01 steps (91 candidates)
- **Early stopping** — saves best model, stops if val stagnates for 3 epochs after epoch 5

In [ ]:
!python train.py \
  --model roberta-base \
  --epochs 8 \
  --batch-size 32 \
  --max-len 128 \
  --focal-loss \
  --focal-gamma 2.0 \
  --label-smoothing 0.03 \
  --fp16

## 4. Inspect results

In [ ]:
import json
from pathlib import Path
model_dir = Path('/content/paychat-model/saved_model')
print('Files in saved_model/:')
for f in sorted(model_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35} {size_kb:>10.1f} KB')

print('\nLearned per-intent thresholds:')
with open(model_dir / 'thresholds.json') as f:
    thresholds = json.load(f)
for intent, thr in thresholds.items():
    print(f'  {intent:<14} {thr:.2f}')

print('\nPer-intent test metrics:')
with open(model_dir / 'training_report.json') as f:
    report = json.load(f)
print(f"  test exact-match: {report['test_exact_match']:.2%}")
print(f"  test hamming:     {report['test_hamming']:.2%}")
print()
print(f"  {'intent':<14} {'precision':>10} {'recall':>8} {'f1':>7}")
for intent, m in report['per_intent'].items():
    print(f"  {intent:<14} {m['precision']:>9.1%} {m['recall']:>7.1%} {m['f1']:>6.1%}")

## 5. Seed-suite regression (the real test)

v3 scored **43/82 (52.4%)**. v4.0 scored **67/82 (81.7%)**. v4.3 scored **~108/110 (98%+)**. v5 expands the suite to **126 cases** with 16 multi-turn context tests.

v5 should reach **≥120/126 (≥95%)** including the new context-dependent tests. If it doesn't beat 95%, look at the failure list — context tests are the hardest since the model is learning a new skill.

The seed test runner now supports a `context` field in test cases. Multi-turn tests use tokenizer pair encoding matching the training format.

In [ ]:
%cd /content/paychat-model
!python eval/run_seed_baseline.py 2>&1 | tail -20

In [ ]:
# Pull the headline numbers from the report so they're easy to compare to v3.
import json
rpt = json.load(open('/content/paychat-model/eval/baseline_report.json'))
print(f"v4 seed-suite: {rpt['passed']}/{rpt['total']} passed ({rpt['passed']/rpt['total']*100:.1f}%)")
print(f"v4 IID test:   {rpt['test_exact_match']:.2f}% exact match")
print(f"v4 latency:    {rpt['ms_per_case']:.0f} ms/case")
print()
print('By tag:')
for tag, info in sorted(rpt['by_tag'].items(), key=lambda x: x[1]['passed']/x[1]['total']):
    rate = info['passed']/info['total']*100
    print(f"  {tag:<25} {info['passed']:>2}/{info['total']:<2}  ({rate:.0f}%)")

## 6. Sanity check — fire real messages through the trained model

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/paychat-model/saved_model'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().cuda()
with open(f'{MODEL_DIR}/thresholds.json') as f:
    thresholds = json.load(f)
labels = list(mdl.config.id2label.values()) if mdl.config.id2label else list(thresholds.keys())

def infer(text, context=None):
    """Run inference with optional context."""
    if context:
        enc = tok(context, text, return_tensors='pt', truncation=True, max_length=128).to('cuda')
    else:
        enc = tok(text, return_tensors='pt', truncation=True, max_length=128).to('cuda')
    with torch.no_grad():
        logits = mdl(**enc).logits[0]
    probs = torch.sigmoid(logits).cpu().tolist()
    fired = [(labels[i], probs[i]) for i in range(len(labels)) if probs[i] >= thresholds.get(labels[i], 0.5)]
    fired.sort(key=lambda x: -x[1])
    return ', '.join(f'{l}={p:.2f}' for l, p in fired) or '(none)'

print("=" * 70)
print("  SINGLE-TURN (v4.3 regressions — must still pass)")
print("=" * 70)
single_turn = [
    'venmo me 20 bucks for pizza',
    'remind me to call mom tomorrow at 6pm',
    'call dad',
    "I'm not paying for pizza",
    'paid rent already',
    'uber to JFK at 5am tomorrow',
    'playing it safe',
    'good call',
    'noted',
    "I'm watching tv",
    "she paid 200 for that bag",
    'uber please',
    'venmo me',
    'uber',
    'I love Paris',
]
for text in single_turn:
    print(f"  {text!r:<55} -> {infer(text)}")

print()
print("=" * 70)
print("  MULTI-TURN CONTEXT (v5 — the new stuff)")
print("=" * 70)
multi_turn = [
    # Context buildup: casual → trigger
    ("bro I'm so hungry | same, haven't eaten all day", "dominos?",           "food buildup"),
    ("I'm running late | the train is delayed",         "uber?",              "ride buildup"),
    ("dinner was 120 total | that's expensive",          "split?",             "money buildup"),
    ("should I bring an umbrella",                       "check the weather?", "weather buildup"),

    # Confirmation after actionable
    ("should we order pizza tonight",                    "yeah do it",         "food confirm"),
    ("uber to the airport?",                             "sure let's go",      "ride confirm"),
    ("venmo me 20 for the pizza",                        "ok sending",         "money confirm"),

    # Confirmation after non-actionable (must be silent!)
    ("that movie was wild",                              "yeah",               "casual confirm NEG"),
    ("I'm so tired today",                               "same",               "casual confirm NEG"),
    ("bro that meme was hilarious",                      "lol yeah",           "casual confirm NEG"),

    # Context negative: same word, different context
    ("what's your favorite app",                         "dominos",            "context NEG"),
    ("their stock went up today",                        "uber",               "context NEG"),
    ("which payment app is best",                        "venmo",              "context NEG"),

    # Correction
    ("uber to the airport",                              "actually make it lyft", "correction"),
    ("venmo me 20 for dinner",                           "actually make it 30",   "correction"),

    # No carryforward: casual after actionable
    ("order pizza from dominos",                         "that was a good game last night", "no carry NEG"),

    # Disambiguation: same word, different outcome
    ("should we order pizza",                            "sure",               "disambig: food"),
    ("that movie was really good",                       "sure",               "disambig: nothing"),

    # Single-turn with context noise (must still work)
    ("what's up | nothing much",                         "venmo me 20 for pizza", "single w/ context"),
]
for ctx, text, desc in multi_turn:
    result = infer(text, context=ctx)
    print(f"  [{desc}]")
    print(f"    context: {ctx!r}")
    print(f"    text:    {text!r:<40} -> {result}")
    print()

## 7. Zip and download `saved_model/`

Drop the unzipped folder into your local repo at `./saved_model/`, then re-run `python eval/run_seed_baseline.py` locally to refresh `eval/baseline_report.md`. Compare to the v3 baseline you already have on disk.

In [ ]:
%cd /content/paychat-model
!zip -r saved_model_v5.zip saved_model/ -x '*.bin.tmp' '*.cache*' > /dev/null
!ls -lh saved_model_v5.zip
from google.colab import files
files.download('saved_model_v5.zip')